# Lab 10 · A minimal eval harness
**~30 minutes · costs about $0.02 · Domain 4, and the gate for Domains 5 and 6**

Domain 4 is only 2.6% of the exam, but evals decide model selection (D5), gate
prompt tuning (D6), and clear migrations (D2). You will see the thing that makes
the case: **a prompt fix that repairs one case and breaks two others.**

In [ ]:
import os, anthropic
client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY
MODEL  = "claude-sonnet-4-6"            # verify against lab 00 output
CHEAP  = "claude-haiku-4-5"             # for high-volume steps
print("sdk", anthropic.__version__)

## A golden set, including the awkward cases

In [ ]:
GOLDEN = [
    ("Refund never arrived after three weeks.",        "billing"),
    ("App crashes on launch since the update.",        "technical"),
    ("How do I change my plan?",                       "account"),
    ("Charged twice this month.",                      "billing"),
    ("Cannot log in, password reset does nothing.",    "technical"),
    ("Please cancel my subscription.",                 "account"),
    ("Invoice shows the wrong VAT rate.",              "billing"),
    ("Export button greys out on large files.",        "technical"),
    ("I was billed after cancelling.",                 "billing"),   # billing vs account
    ("Reset link expired before I clicked it.",        "technical"), # technical vs account
]
print(len(GOLDEN), "cases")

## Code-based grading

One right answer per case, so exact match is the correct grader: cheapest,
fastest, no drift. Reach for an LLM judge only when exact match cannot apply.

In [ ]:
def classify(text, prompt):
    r = client.messages.create(model=CHEAP, max_tokens=10, system=prompt,
            messages=[{"role":"user","content":text}])
    return r.content[0].text.strip().lower().strip(".")

def evaluate(prompt, label):
    results, correct = [], 0
    for text, expected in GOLDEN:
        got = classify(text, prompt)
        ok = got == expected
        correct += ok
        results.append((text, expected, got, ok))
    print(f"{label:<12} {correct}/{len(GOLDEN)}  ({correct/len(GOLDEN):.0%})")
    return results, correct

## Baseline

In [ ]:
V1 = ("Classify the support ticket into exactly one category: "
      "billing, technical, or account. Reply with the single word only.")
r1, s1 = evaluate(V1, "v1")
for t,e,g,ok in r1:
    if not ok: print(f"   MISS {t[:44]:<46} expected={e:<10} got={g}")

## Fix a specific failure

Add a rule targeting the billing-versus-account confusion. This is exactly how
prompt tuning happens in the wild: someone reports a case, you patch it.

In [ ]:
V2 = (V1 + " Anything involving a charge, refund, or invoice is billing, "
      "even if it also mentions cancelling or an account.")
r2, s2 = evaluate(V2, "v2")

## Did the patch trade one failure for another?

In [ ]:
before = {t: ok for t,_,_,ok in r1}
after  = {t: ok for t,_,_,ok in r2}
fixed  = [t for t in before if not before[t] and after[t]]
broke  = [t for t in before if before[t] and not after[t]]

print(f"score {s1} -> {s2}")
print("\nFIXED:");  [print("   +", t[:60]) for t in fixed]  or print("   (none)")
print("\nREGRESSED:"); [print("   -", t[:60]) for t in broke] or print("   (none)")
print()
print("Without the golden set you would only have seen the case you were looking at.")
print("That blindness is the entire argument for building the eval set FIRST.")

## When exact match will not do

For open-ended output you need a judge — and **the judge itself must be
calibrated against human labels.** A judge produces consistent scores whether or
not it measures the thing you care about. Consistency is not accuracy.

In [ ]:
def judge(answer, criterion):
    r = client.messages.create(model=MODEL, max_tokens=120,
        system=("You are grading one dimension only. Reply with a score 1-5 and one "
                "short reason. If you cannot tell from what you were given, reply UNKNOWN."),
        messages=[{"role":"user","content":f"CRITERION: {criterion}\n\nANSWER:\n{answer}"}])
    return r.content[0].text.strip()

print(judge("Your refund was processed on Tuesday and should appear in 3-5 days.",
            "Does the answer give the customer a concrete next step and timeframe?"))
print()
print("Grade ONE dimension per judge. Give it an UNKNOWN escape hatch.")
print("Then validate it: run it over human-labelled cases and measure agreement.")

---
### Checkpoint
- Rank the grading methods from cheapest to most expensive
- Why must an LLM judge be validated, and against what?
- Which production signal reveals a silent regression that no error dashboard catches?